In [2]:
r"""
=================================================================================
ORALVISION-M4 — FULL-BUDGET ABLATION STUDY (CLEAN DATASET)
RTX 2050 HARDENED + TRUE PER-EPOCH RESUME + REPRODUCIBILITY REPORT
=================================================================================

This is the FULL-BUDGET successor to your original ablation script
(m4-ablation_study.ipynb / m4_coordatt_logreg-ab-study.zip). Same 13
variant definitions, same architecture code (ConfigurableCoordAtt +
EfficientNetCoordAtt), same variant naming — so results stay directly
comparable to your original Table 5/6 in the manuscript.

*** PATCH NOTE (applied) ***
Variant 7 ("No Class Weighting") previously used sampler_mode="random",
which disabled ALL class-based oversampling for the entire 40-epoch run
instead of isolating only the loss-weighting term as the manuscript's
Section 4.2 text describes. This has been changed to sampler_mode=
"difficulty", so Variant 7 now uses the exact same class-balanced +
difficulty-weighted sampler as the Baseline during warmup/cascade, and
differs from the Baseline ONLY in class_weighted_loss=False. This is the
only change in this file relative to the version you were already
running. Because the run had not yet reached Variant 7 when this was
applied, no existing checkpoint needs to be deleted — Variant 1 (and 2,
5, 6 when their turn comes) resume/run exactly as before; Variant 7
starts fresh with the corrected sampler; Variants 8 and 13 are unchanged
(8 was already correctly "random" by design, 13 never used "random").

WHAT CHANGED VS. YOUR ORIGINAL ABLATION SCRIPT
-----------------------------------------------
1. FULL TRAINING BUDGET
   Original ablation:  5+10 curriculum epochs, or 15 single-phase epochs.
   This version:       8+32 curriculum epochs (=40), or 40 single-phase —
                        IDENTICAL budget to the main M4 training run, so
                        this directly answers the reviewer's Tier-1 item:
                        "re-run key ablations under a controlled training
                        budget; do not rely on reduced-budget comparisons."

2. DATASET
   Original ablation rebuilt a group-aware split from the merged dataset
   on the fly (orig/beng pairing). This version instead consumes the
   ALREADY-CORRECTED, already-split oral_dataset_FINAL_CLEAN directory
   (train/val/test folders with augmentation applied to train only,
   val/test held out untouched) — i.e. it trains on the leakage-audited
   dataset, not the old merged one.

3. TRUE PER-EPOCH CHECKPOINT + RESUME
   Your original script could only retry an ENTIRE variant from scratch
   if CUDA failed. This version checkpoints EVERY epoch, atomically, per
   variant, and on restart resumes from the exact epoch it stopped on —
   model, optimizer, scheduler, AMP scaler, difficulty dict, best-state,
   and full RNG state are all restored. A variant whose final_result.json
   already exists is skipped entirely, so you can also add variants later
   without re-running completed ones.

4. RTX 2050 HARDENING
   AMP (mixed precision), gradient accumulation (real batch 16 -> effective
   32), expandable_segments CUDA allocator, explicit GPU memory logging,
   and an OOM-safe training step that cleans up and re-raises with an
   actionable message rather than silently corrupting state.

5. REPRODUCIBILITY REPORT
   Writes reproducibility_report.txt / .json capturing: Python/torch/
   torchvision/CUDA/cuDNN versions, OS/CPU/GPU info, key library versions
   (numpy, scikit-learn, Pillow, thop), the exact random seed, the exact
   checkpoint-selection rule in plain English, the FLOPs tool used and
   its version, and the dataset paths/counts.

OUTPUT
------
OUTPUT_ROOT/
    reproducibility_report.txt
    reproducibility_report.json
    <safe_variant_name>/
        checkpoint.pth              <- resumable, every epoch
        best_model.pth
        training_config.json
        history.json
        final_result.json           <- presence of this file == "done"
    ablation_results_full_budget.csv
    ablation_results_full_budget.json
    ablation_results_full_budget.txt
=================================================================================
"""

import os

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:128",
)

import gc
import csv
import json
import copy
import time
import random
import platform
import warnings
import subprocess
from pathlib import Path
from datetime import datetime
from importlib.metadata import version as pkg_version, PackageNotFoundError

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, RandomSampler
from torch.amp import autocast, GradScaler

import torchvision
from torchvision import transforms, models

from PIL import Image

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 1. PATHS — EDIT THESE TWO FOR YOUR MACHINE
# =============================================================================

DATASET_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN"
)

OUTPUT_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\m4_ablation_full_budget_clean"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# 2. FIXED SETTINGS
# =============================================================================

SEED = 42
IMG_SIZE = 224

# RTX 2050 (4GB) safe configuration
REAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = REAL_BATCH_SIZE * GRAD_ACCUM_STEPS  # 32
NUM_WORKERS = 0
USE_AMP = True

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DIFFICULTY_ALPHA = 1.5

# FULL BUDGET — identical schedule to the main M4 training run
FULL_WARMUP_EPOCHS = 8
FULL_CASCADE_EPOCHS = 32
FULL_CURRICULUM_EPOCHS = FULL_WARMUP_EPOCHS + FULL_CASCADE_EPOCHS  # 40
FULL_SINGLE_PHASE_EPOCHS = 40

EARLY_STOP_PATIENCE = 10
LR_PATIENCE = 3
LR_FACTOR = 0.5
MIN_LR = 1e-7

# Exact, explicit checkpoint-selection rule (also written into the
# reproducibility report verbatim — this is the answer to "how was the
# reported checkpoint chosen" that reviewers ask for)
CHECKPOINT_SELECTION_RULE = (
    "For each variant, the checkpoint reported is the one with the highest "
    "validation macro-F1 observed at any epoch during that variant's training "
    "run (full budget: 8 warm-up + 32 cascade epochs for curriculum variants, "
    "40 single-phase epochs otherwise). Ties are broken by the earliest epoch "
    "achieving that value. Training additionally stops early if validation "
    "macro-F1 does not improve for 10 consecutive epochs. The selected "
    "checkpoint's weights (not the final-epoch weights) are what is loaded "
    "for test-set evaluation and reported in all result tables."
)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# =============================================================================
# 3. LOGGING / GPU HELPERS
# =============================================================================


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        except Exception:
            pass


def print_gpu_memory(prefix=""):
    if not torch.cuda.is_available():
        return
    try:
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        free_mem, total_mem = torch.cuda.mem_get_info()
        log(
            f"{prefix}GPU allocated={allocated:.2f}GB reserved={reserved:.2f}GB "
            f"free={free_mem/1e9:.2f}GB/{total_mem/1e9:.2f}GB"
        )
    except Exception:
        pass


def safe_filename(name):
    return (
        name.replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
        .replace("=", "")
        .replace("/", "-")
        .replace("+", "plus")
    )


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state().detach().cpu().to(torch.uint8).contiguous(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = [
            s.detach().cpu().to(torch.uint8).contiguous()
            for s in torch.cuda.get_rng_state_all()
        ]
    return state


def set_rng_state(state):
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        t = state["torch"]
        if not isinstance(t, torch.Tensor):
            t = torch.tensor(t, dtype=torch.uint8, device="cpu")
        torch.set_rng_state(t.detach().cpu().to(torch.uint8).contiguous())
    if torch.cuda.is_available() and "torch_cuda" in state:
        fixed = []
        for s in state["torch_cuda"]:
            if not isinstance(s, torch.Tensor):
                s = torch.tensor(s, dtype=torch.uint8, device="cpu")
            fixed.append(s.detach().cpu().to(torch.uint8).contiguous())
        if fixed:
            torch.cuda.set_rng_state_all(fixed)


# =============================================================================
# 4. REPRODUCIBILITY REPORT
# =============================================================================


def _safe_pkg_version(pkg_name, import_name=None):
    try:
        return pkg_version(pkg_name)
    except PackageNotFoundError:
        pass
    try:
        mod = __import__(import_name or pkg_name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not installed"


def _git_commit():
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        )
        return out.decode().strip()
    except Exception:
        return "not a git repo / git unavailable"


def gather_reproducibility_info(dataset_info):
    gpu_name = "N/A (CPU run)"
    gpu_mem_gb = None
    cuda_version = "N/A"
    cudnn_version = "N/A"

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        cuda_version = torch.version.cuda
        cudnn_version = str(torch.backends.cudnn.version())

    info = {
        "generated": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "random_seed": SEED,
        "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
        "software": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "torchvision": torchvision.__version__,
            "cuda": cuda_version,
            "cudnn": cudnn_version,
            "numpy": _safe_pkg_version("numpy"),
            "scikit_learn": _safe_pkg_version("scikit-learn", "sklearn"),
            "pillow": _safe_pkg_version("Pillow", "PIL"),
            "thop": _safe_pkg_version("thop"),
        },
        "hardware": {
            "os": f"{platform.system()} {platform.release()} ({platform.version()})",
            "cpu": platform.processor() or platform.machine(),
            "gpu": gpu_name,
            "gpu_memory_gb": round(gpu_mem_gb, 2) if gpu_mem_gb else None,
        },
        "training_config": {
            "real_batch_size": REAL_BATCH_SIZE,
            "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
            "effective_batch_size": EFFECTIVE_BATCH_SIZE,
            "amp_enabled": USE_AMP,
            "img_size": IMG_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "difficulty_alpha": DIFFICULTY_ALPHA,
            "full_warmup_epochs": FULL_WARMUP_EPOCHS,
            "full_cascade_epochs": FULL_CASCADE_EPOCHS,
            "full_curriculum_epochs_total": FULL_CURRICULUM_EPOCHS,
            "full_single_phase_epochs": FULL_SINGLE_PHASE_EPOCHS,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "lr_schedule": "ReduceLROnPlateau(mode=min, factor=0.5, patience=3, min_lr=1e-7)",
        },
        "flops_tool": "thop.profile (see compute_model_flops()); reported as 2x measured MACs",
        "dataset": dataset_info,
        "git_commit": _git_commit(),
    }
    return info


def write_reproducibility_report(info):
    json_path = OUTPUT_ROOT / "reproducibility_report.json"
    txt_path = OUTPUT_ROOT / "reproducibility_report.txt"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)

    lines = [
        "OralVision-M4 — Reproducibility Report (Full-Budget Ablation Run)",
        "=" * 78,
        f"Generated: {info['generated']}",
        f"Random seed: {info['random_seed']}",
        "",
        "CHECKPOINT SELECTION RULE",
        "-" * 78,
        info["checkpoint_selection_rule"],
        "",
        "SOFTWARE VERSIONS",
        "-" * 78,
    ]
    for k, v in info["software"].items():
        lines.append(f"  {k:15s}: {v}")

    lines += ["", "HARDWARE", "-" * 78]
    for k, v in info["hardware"].items():
        lines.append(f"  {k:15s}: {v}")

    lines += ["", "TRAINING CONFIGURATION", "-" * 78]
    for k, v in info["training_config"].items():
        lines.append(f"  {k:32s}: {v}")

    lines += [
        "",
        "FLOPS COMPUTATION",
        "-" * 78,
        f"  {info['flops_tool']}",
        "",
        "DATASET",
        "-" * 78,
    ]
    for k, v in info["dataset"].items():
        lines.append(f"  {k:15s}: {v}")

    lines += [
        "",
        "VERSION CONTROL",
        "-" * 78,
        f"  git commit: {info['git_commit']}",
    ]

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    log(f"Reproducibility report written to:\n  {txt_path}\n  {json_path}")


def compute_model_flops(model, img_size=IMG_SIZE):
    """Uses thop, matching the tool named in the reproducibility report."""
    try:
        from thop import profile

        was_training = model.training
        model.eval()
        dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)
        with torch.no_grad():
            macs, params = profile(model, inputs=(dummy,), verbose=False)
        flops = float(macs) * 2
        if was_training:
            model.train()
        del dummy
        cleanup_gpu()
        return flops, params
    except ImportError:
        log("thop not installed — FLOPs will be N/A. `pip install thop` to enable.")
        return None, None
    except Exception as exc:
        log(f"FLOPs computation failed: {exc}")
        return None, None


# =============================================================================
# 5. DATASET — consumes the already-corrected, already-split clean dataset
# =============================================================================

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset directory not found:\n{DATASET_ROOT}")


def find_split_directory(root, names):
    if isinstance(names, str):
        names = [names]
    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        if p.name.lower() in {x.lower() for x in names}:
            class_dirs = [x for x in p.iterdir() if x.is_dir()]
            if len(class_dirs) >= 2:
                candidates.append((len(class_dirs), p))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


TRAIN_DIR = find_split_directory(DATASET_ROOT, "train")
VAL_DIR = find_split_directory(DATASET_ROOT, ["val", "validation"])
TEST_DIR = find_split_directory(DATASET_ROOT, "test")

for _n, _d in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    if _d is None:
        raise FileNotFoundError(f"Could not find a {_n.upper()} directory under {DATASET_ROOT}")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def discover_classes(train_dir, val_dir, test_dir):
    tc = {p.name for p in train_dir.iterdir() if p.is_dir()}
    vc = {p.name for p in val_dir.iterdir() if p.is_dir()}
    sc = {p.name for p in test_dir.iterdir() if p.is_dir()}
    if tc != vc or tc != sc:
        raise RuntimeError(
            f"Class mismatch across splits.\nTrain:{sorted(tc)}\nVal:{sorted(vc)}\nTest:{sorted(sc)}"
        )
    return sorted(tc)


CLASS_NAMES = discover_classes(TRAIN_DIR, VAL_DIR, TEST_DIR)
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: n for n, i in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_NAMES)


def collect_samples(split_dir):
    samples = []
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                samples.append((str(path), CLASS_TO_IDX[class_name]))
    return samples


TRAIN_SAMPLES = collect_samples(TRAIN_DIR)
VAL_SAMPLES = collect_samples(VAL_DIR)
TEST_SAMPLES = collect_samples(TEST_DIR)

log(f"Dataset sizes -> train={len(TRAIN_SAMPLES)}, val={len(VAL_SAMPLES)}, test={len(TEST_SAMPLES)}")
log(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

TRAIN_LABELS_ALL = np.array([lbl for _, lbl in TRAIN_SAMPLES], dtype=np.int64)

CLASS_WEIGHTS_BALANCED = {
    int(i): float(w)
    for i, w in zip(
        np.arange(NUM_CLASSES),
        compute_class_weight(
            class_weight="balanced", classes=np.arange(NUM_CLASSES), y=TRAIN_LABELS_ALL
        ),
    )
}


class OralDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as img:
            img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label, index


eval_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

aug_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

val_dataset = OralDataset(VAL_SAMPLES, transform=eval_transform)
test_dataset = OralDataset(TEST_SAMPLES, transform=eval_transform)


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=REAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )


val_loader = make_eval_loader(val_dataset)
test_loader = make_eval_loader(test_dataset)

# =============================================================================
# 6. MODEL — identical to your original ablation script's architecture
# =============================================================================


class ConfigurableCoordAtt(nn.Module):
    """
    attention_mode: 'full' | 'height_only' | 'width_only' | 'none'
    Identical semantics to your original ablation script.
    """

    def __init__(self, channels, reduction=32, attention_mode="full"):
        super().__init__()
        self.attention_mode = attention_mode
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, channels // reduction)
        self.conv1 = nn.Conv2d(channels, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        if self.attention_mode == "none":
            return x

        identity = x
        _, _, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))

        if self.attention_mode == "full":
            return identity * a_h * a_w
        elif self.attention_mode == "height_only":
            return identity * a_h
        elif self.attention_mode == "width_only":
            return identity * a_w
        else:
            raise ValueError(f"Unknown attention_mode: {self.attention_mode}")


class EfficientNetCoordAtt(nn.Module):
    """classifier_head: 'logreg' | 'linear_dropout' — same as your original script."""

    def __init__(
        self,
        num_classes,
        ca_reduction=32,
        dropout=0.3,
        attention_mode="full",
        classifier_head="logreg",
        pretrained=True,
    ):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        base = models.efficientnet_b0(weights=weights)
        self.features = base.features
        self.coordatt = ConfigurableCoordAtt(1280, ca_reduction, attention_mode)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        if classifier_head == "logreg":
            self.classifier = nn.Linear(1280, num_classes)
        elif classifier_head == "linear_dropout":
            self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, num_classes))
        else:
            raise ValueError(f"Unknown classifier_head: {classifier_head}")

    def forward(self, x):
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# =============================================================================
# 7. VARIANT DEFINITIONS — identical names/configs to your original Table 5
#    *** except Variant 7's sampler_mode, which is patched below ***
# =============================================================================

ALL_VARIANTS = [
    dict(
        name="1. Baseline (Full Model)",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="2. No Attention (CoordAtt removed)",
        attention_mode="none", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="3. Height-Only Attention",
        attention_mode="height_only", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="4. Width-Only Attention",
        attention_mode="width_only", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="5. No Curriculum (single-phase)",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=False, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="6. No Hard-Example Cascade",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="class_only", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        # *** PATCHED: sampler_mode was "random", now "difficulty" ***
        # This isolates ONLY the loss-weighting term (class_weighted_loss=False)
        # while keeping the same class-balanced + difficulty-weighted sampler
        # as the Baseline during warmup/cascade — matching what Section 4.2 of
        # the manuscript describes this variant as testing.
        name="7. No Class Weighting",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=False,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="8. Naive Training (no CoordAtt, no curriculum, no weighting)",
        attention_mode="none", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=False, sampler_mode="random", class_weighted_loss=False,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="9. CoordAtt Reduction r=16",
        attention_mode="full", ca_reduction=16, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="10. CoordAtt Reduction r=8",
        attention_mode="full", ca_reduction=8, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
    dict(
        name="11. No Data Augmentation",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=False, lr_schedule=True,
    ),
    dict(
        name="12. Fixed LR (no ReduceLROnPlateau)",
        attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=False,
    ),
    dict(
        name="13. Linear+Dropout Head (instead of Logistic Regression)",
        attention_mode="full", ca_reduction=32, classifier_head="linear_dropout", dropout=0.3,
        curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
        augmentation=True, lr_schedule=True,
    ),
]

# -----------------------------------------------------------------------
# Which variants to actually run at FULL BUDGET this pass.
# Per your reviewer's own guidance, the "key" variants are the ones that
# drove the contested finding in Section 4.2/4.3 of the manuscript.
# Uncomment more names below (or set RUN_VARIANT_NAMES = None to run all 13)
# if you have the GPU time — each full-budget variant is ~1.5-2.5 hours.
# -----------------------------------------------------------------------
RUN_VARIANT_NAMES = {
    "1. Baseline (Full Model)",
    "2. No Attention (CoordAtt removed)",
    "5. No Curriculum (single-phase)",
    "6. No Hard-Example Cascade",
    "7. No Class Weighting",
    "8. Naive Training (no CoordAtt, no curriculum, no weighting)",
    "13. Linear+Dropout Head (instead of Logistic Regression)",
    # "3. Height-Only Attention",
    # "4. Width-Only Attention",
    # "9. CoordAtt Reduction r=16",
    # "10. CoordAtt Reduction r=8",
    # "11. No Data Augmentation",
    # "12. Fixed LR (no ReduceLROnPlateau)",
}

VARIANTS = (
    ALL_VARIANTS
    if RUN_VARIANT_NAMES is None
    else [v for v in ALL_VARIANTS if v["name"] in RUN_VARIANT_NAMES]
)

KEY_VARIANTS_FOR_PER_CLASS = {
    "1. Baseline (Full Model)",
    "7. No Class Weighting",
    "8. Naive Training (no CoordAtt, no curriculum, no weighting)",
}

# =============================================================================
# 8. SAMPLER
# =============================================================================


def build_sampler(mode, difficulty_dict=None):
    if mode == "random":
        return RandomSampler(range(len(TRAIN_SAMPLES)))

    weights = []
    for i, (_, label) in enumerate(TRAIN_SAMPLES):
        w = CLASS_WEIGHTS_BALANCED[label]
        if mode == "difficulty" and difficulty_dict is not None:
            diff = difficulty_dict.get(i, 1.0)
            diff = max(0.25, min(float(diff), 10.0))
            w *= diff ** DIFFICULTY_ALPHA
        weights.append(w)
    weights = np.maximum(np.asarray(weights, dtype=np.float64), 1e-12)
    weights = weights / weights.sum() * len(weights)
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
    )


# =============================================================================
# 9. TRAIN / EVAL LOOPS — AMP + gradient accumulation + OOM-safe
# =============================================================================


def run_train_epoch(model, loader, optimizer, scaler, criterion, difficulty_dict, update_difficulty):
    model.train()
    running_loss = 0.0
    total = 0
    all_labels, all_preds = [], []

    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels, indices) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        is_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(loader)

        try:
            with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
                outputs = model(images)
                per_sample_loss = nn.functional.cross_entropy(
                    outputs, labels, weight=criterion.weight, reduction="none"
                )
                raw_loss = per_sample_loss.mean()
                loss = raw_loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()

        except torch.cuda.OutOfMemoryError:
            log(f"!! CUDA OOM at batch {step + 1}/{len(loader)}. Cleaning up and re-raising.")
            optimizer.zero_grad(set_to_none=True)
            cleanup_gpu()
            print_gpu_memory("After OOM cleanup: ")
            log("If this repeats, lower REAL_BATCH_SIZE or raise GRAD_ACCUM_STEPS.")
            raise

        if update_difficulty:
            with torch.no_grad():
                losses_np = per_sample_loss.detach().float().cpu().numpy()
                for gi, sample_loss in zip(indices.numpy(), losses_np):
                    gi = int(gi)
                    prev = difficulty_dict.get(gi, 1.0)
                    difficulty_dict[gi] = 0.5 * prev + 0.5 * float(sample_loss)

        with torch.no_grad():
            preds = torch.argmax(outputs, dim=1)
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_preds.extend(preds.detach().cpu().numpy().tolist())

        bs = images.size(0)
        running_loss += raw_loss.item() * bs
        total += bs

        if is_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        del images, labels, outputs, per_sample_loss, raw_loss, loss

    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return {"loss": running_loss / max(total, 1), "macro_f1": float(f1)}


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    total = 0
    all_labels, all_preds = [], []

    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels_device = labels.to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels_device)

        preds = torch.argmax(outputs, dim=1)
        bs = images.size(0)
        running_loss += loss.item() * bs
        total += bs

        all_labels.extend(labels.numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

        del images, labels_device, outputs, loss, preds

    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    acc = accuracy_score(all_labels, all_preds)
    return (
        {"loss": running_loss / max(total, 1), "macro_f1": float(f1), "accuracy": float(acc)},
        np.asarray(all_labels),
        np.asarray(all_preds),
    )


# =============================================================================
# 10. PER-VARIANT DRIVER — resumable at exact epoch
# =============================================================================


def run_variant(cfg):
    name = cfg["name"]
    variant_dir = OUTPUT_ROOT / safe_filename(name)
    variant_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = variant_dir / "checkpoint.pth"
    best_model_path = variant_dir / "best_model.pth"
    result_path = variant_dir / "final_result.json"

    if result_path.exists():
        log(f"[{name}] Already completed — skipping.")
        with open(result_path, "r", encoding="utf-8") as f:
            return json.load(f)

    log("=" * 78)
    log(f"VARIANT: {name}")
    log(f"Config: {cfg}")
    log("=" * 78)

    seed_everything(SEED)

    total_epochs = FULL_CURRICULUM_EPOCHS if cfg["curriculum"] else FULL_SINGLE_PHASE_EPOCHS
    warmup_epochs = FULL_WARMUP_EPOCHS if cfg["curriculum"] else 0

    with open(variant_dir / "training_config.json", "w", encoding="utf-8") as f:
        json.dump({**cfg, "total_epochs": total_epochs, "seed": SEED}, f, indent=2)

    train_transform = aug_transform if cfg["augmentation"] else eval_transform
    train_dataset = OralDataset(TRAIN_SAMPLES, transform=train_transform)

    model = EfficientNetCoordAtt(
        num_classes=NUM_CLASSES,
        ca_reduction=cfg["ca_reduction"],
        dropout=cfg["dropout"],
        attention_mode=cfg["attention_mode"],
        classifier_head=cfg["classifier_head"],
        pretrained=True,
    ).to(DEVICE)

    if cfg["class_weighted_loss"]:
        class_weight_tensor = torch.tensor(
            [CLASS_WEIGHTS_BALANCED[i] for i in range(NUM_CLASSES)], dtype=torch.float32, device=DEVICE
        )
    else:
        class_weight_tensor = torch.ones(NUM_CLASSES, dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    scheduler = None
    if cfg["lr_schedule"]:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR
        )

    scaler = GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

    difficulty_dict = {}
    history = {"train_loss": [], "train_macro_f1": [], "val_loss": [], "val_macro_f1": [], "lr": []}
    best_val_f1 = -np.inf
    best_epoch = 0
    best_state = None
    patience_counter = 0
    start_epoch = 1

    # -------------------- RESUME --------------------
    if checkpoint_path.exists():
        log(f"[{name}] Resuming from checkpoint.")
        ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        history = ckpt["history"]
        best_val_f1 = ckpt["best_val_macro_f1"]
        best_epoch = ckpt["best_epoch"]
        best_state = ckpt["best_state"]
        patience_counter = ckpt["patience_counter"]
        difficulty_dict.clear()
        difficulty_dict.update(ckpt.get("difficulty_dict", {}))
        try:
            set_rng_state(ckpt["rng_state"])
        except Exception as exc:
            log(f"[{name}] RNG restore failed (continuing anyway): {exc}")
        start_epoch = int(ckpt["epoch"]) + 1
        log(f"[{name}] Resuming at epoch {start_epoch}/{total_epochs}, best_val_f1={best_val_f1:.6f}")

    def save_ckpt(epoch):
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
            "best_val_macro_f1": best_val_f1,
            "best_epoch": best_epoch,
            "best_state": best_state,
            "patience_counter": patience_counter,
            "difficulty_dict": difficulty_dict,
            "rng_state": get_rng_state(),
        }
        tmp = variant_dir / "checkpoint.tmp"
        torch.save(ckpt, tmp)
        tmp.replace(checkpoint_path)

    # -------------------- TRAIN --------------------
    variant_start_time = time.time()

    for epoch in range(start_epoch, total_epochs + 1):
        epoch_start = time.time()

        in_cascade = cfg["curriculum"] and epoch > warmup_epochs
        stage = "CASCADE" if in_cascade else ("WARMUP" if cfg["curriculum"] else "SINGLE-PHASE")

        if cfg["sampler_mode"] == "difficulty":
            effective_mode = "difficulty" if in_cascade else "class_only"
        else:
            effective_mode = cfg["sampler_mode"]

        sampler = build_sampler(effective_mode, difficulty_dict)
        train_loader = DataLoader(
            train_dataset,
            batch_size=REAL_BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        )

        update_difficulty = cfg["sampler_mode"] == "difficulty" and in_cascade

        train_metrics = run_train_epoch(
            model, train_loader, optimizer, scaler, criterion, difficulty_dict, update_difficulty
        )
        cleanup_gpu()

        val_metrics, _, _ = evaluate_loader(model, val_loader, criterion)
        cleanup_gpu()

        if scheduler is not None:
            scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - epoch_start

        history["train_loss"].append(train_metrics["loss"])
        history["train_macro_f1"].append(train_metrics["macro_f1"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["lr"].append(current_lr)

        log(
            f"[{name}] [{stage}] Epoch {epoch:03d}/{total_epochs} | "
            f"train_loss={train_metrics['loss']:.4f} train_F1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_F1={val_metrics['macro_f1']:.4f} | "
            f"lr={current_lr:.2e} | time={epoch_time:.1f}s"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)
            patience_counter = 0
            log(f"[{name}]   -> NEW BEST (val macro-F1={best_val_f1:.6f})")
        else:
            patience_counter += 1

        save_ckpt(epoch)
        print_gpu_memory("  ")

        if patience_counter >= EARLY_STOP_PATIENCE:
            log(f"[{name}] Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

    variant_runtime = time.time() - variant_start_time

    # -------------------- FINAL TEST EVAL --------------------
    if best_state is None:
        if best_model_path.exists():
            best_state = torch.load(best_model_path, map_location=DEVICE, weights_only=True)
        else:
            raise RuntimeError(f"[{name}] No best model state available.")

    model.load_state_dict(best_state)
    model.eval()
    cleanup_gpu()

    test_metrics, test_labels, test_preds = evaluate_loader(model, test_loader, criterion)
    prec, rec, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="macro", zero_division=0)
    _, _, per_class_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0
    )
    cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES)))

    flops, thop_params = (None, None)
    if name.startswith("1."):
        # Only compute FLOPs once, on the baseline — matches the main
        # manuscript's model-complexity table and keeps this cheap.
        flops, thop_params = compute_model_flops(model)

    result = {
        "name": name,
        "config": cfg,
        "best_epoch": best_epoch,
        "best_val_macro_f1": float(best_val_f1),
        "accuracy": float(test_metrics["accuracy"]),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "per_class_f1": {CLASS_NAMES[i]: float(per_class_f1[i]) for i in range(NUM_CLASSES)},
        "confusion_matrix": cm.tolist(),
        "runtime_sec": variant_runtime,
        "flops": flops,
        "flops_params": thop_params,
    }

    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    with open(variant_dir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    log(f"[{name}] FINAL — acc={result['accuracy']:.4f} macro_f1={result['f1']:.4f} runtime={variant_runtime:.1f}s")

    del model
    cleanup_gpu()
    return result


# =============================================================================
# 11. MAIN
# =============================================================================


def main():
    log("=" * 78)
    log("ORALVISION-M4 — FULL-BUDGET ABLATION (RTX 2050 HARDENED, RESUMABLE)")
    log("=" * 78)
    log(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        log(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB)")
        print_gpu_memory("Initial: ")
    log(f"Variants queued this run: {[v['name'] for v in VARIANTS]}")
    log(
        f"Full budget: curriculum variants = {FULL_WARMUP_EPOCHS}+{FULL_CASCADE_EPOCHS} "
        f"epochs, single-phase variants = {FULL_SINGLE_PHASE_EPOCHS} epochs "
        f"(early stop patience={EARLY_STOP_PATIENCE})"
    )

    dataset_info = {
        "dataset_root": str(DATASET_ROOT),
        "train_samples": len(TRAIN_SAMPLES),
        "val_samples": len(VAL_SAMPLES),
        "test_samples": len(TEST_SAMPLES),
        "classes": CLASS_NAMES,
    }
    repro_info = gather_reproducibility_info(dataset_info)
    write_reproducibility_report(repro_info)

    all_results = {}
    for cfg in VARIANTS:
        result = run_variant(cfg)
        all_results[cfg["name"]] = result

    if "1. Baseline (Full Model)" not in all_results:
        log("WARNING: baseline not in this run's variant set — deltas will be skipped.")
        baseline_f1 = None
    else:
        baseline_f1 = all_results["1. Baseline (Full Model)"]["f1"]

    # -------------------- summary CSV --------------------
    csv_path = OUTPUT_ROOT / "ablation_results_full_budget.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["variant", "accuracy", "precision", "recall", "f1", "delta_f1_vs_baseline",
             "best_epoch", "runtime_sec"]
        )
        for cfg in VARIANTS:
            r = all_results[cfg["name"]]
            delta = (baseline_f1 - r["f1"]) if baseline_f1 is not None else ""
            writer.writerow(
                [
                    r["name"], f"{r['accuracy']:.4f}", f"{r['precision']:.4f}",
                    f"{r['recall']:.4f}", f"{r['f1']:.4f}",
                    f"{delta:.4f}" if delta != "" else "",
                    r["best_epoch"], f"{r['runtime_sec']:.1f}",
                ]
            )
    log(f"Wrote {csv_path}")

    # -------------------- summary JSON (paper-table format) --------------------
    json_path = OUTPUT_ROOT / "ablation_results_full_budget.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"variants": all_results, "budget": "full (8+32 or 40 single-phase)"}, f, indent=2)
    log(f"Wrote {json_path}")

    # -------------------- per-class breakdown for key variants --------------------
    per_class_path = OUTPUT_ROOT / "per_class_f1_key_variants.csv"
    key_names = [n for n in KEY_VARIANTS_FOR_PER_CLASS if n in all_results]
    if key_names:
        with open(per_class_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["class"] + key_names)
            for class_name in CLASS_NAMES:
                row = [class_name] + [f"{all_results[n]['per_class_f1'][class_name]:.4f}" for n in key_names]
                writer.writerow(row)
        log(f"Wrote {per_class_path}")

    # -------------------- human-readable txt summary --------------------
    txt_path = OUTPUT_ROOT / "ablation_results_full_budget.txt"
    lines = [
        "OralVision-M4 — Full-Budget Ablation Results (Clean/Corrected Dataset)",
        "=" * 90,
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Budget: curriculum variants = {FULL_WARMUP_EPOCHS}+{FULL_CASCADE_EPOCHS} epochs, "
        f"single-phase = {FULL_SINGLE_PHASE_EPOCHS} epochs, early-stop patience = {EARLY_STOP_PATIENCE}",
        "",
        f"{'Variant':60s} {'Acc':>8s} {'Prec':>8s} {'Rec':>8s} {'F1':>8s} {'dF1':>8s} {'Epoch':>7s} {'Runtime(s)':>11s}",
        "-" * 120,
    ]
    for cfg in VARIANTS:
        r = all_results[cfg["name"]]
        delta = (baseline_f1 - r["f1"]) if baseline_f1 is not None else float("nan")
        lines.append(
            f"{r['name']:60s} {r['accuracy']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f} "
            f"{r['f1']:>8.4f} {delta:>8.4f} {r['best_epoch']:>7d} {r['runtime_sec']:>11.1f}"
        )
    lines += [
        "-" * 120,
        "",
        "Delta F1 = F1_baseline - F1_variant (positive = component helps; "
        "negative = variant outperformed baseline).",
        "",
        f"Checkpoint selection rule: {CHECKPOINT_SELECTION_RULE}",
    ]
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    log(f"Wrote {txt_path}")

    log("=" * 78)
    log("ALL QUEUED VARIANTS COMPLETE")
    log("=" * 78)
    for cfg in VARIANTS:
        r = all_results[cfg["name"]]
        d = (baseline_f1 - r["f1"]) if baseline_f1 is not None else float("nan")
        log(f"{r['name']:60s} acc={r['accuracy']:.4f} f1={r['f1']:.4f} delta={d:+.4f}")


if __name__ == "__main__":
    main()

[07:36:06] Dataset sizes -> train=17700, val=1265, test=1265
[07:36:06] Classes (6): ['Calculus', 'Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
[07:36:06] ==============================================================================
[07:36:06] ORALVISION-M4 — FULL-BUDGET ABLATION (RTX 2050 HARDENED, RESUMABLE)
[07:36:06] ==============================================================================
[07:36:06] Device: cuda
[07:36:06] GPU: NVIDIA GeForce RTX 2050 (4.29 GB)
[07:36:06] Initial: GPU allocated=0.82GB reserved=0.85GB free=2.56GB/4.29GB
[07:36:06] Variants queued this run: ['1. Baseline (Full Model)', '2. No Attention (CoordAtt removed)', '5. No Curriculum (single-phase)', '6. No Hard-Example Cascade', '7. No Class Weighting', '8. Naive Training (no CoordAtt, no curriculum, no weighting)', '13. Linear+Dropout Head (instead of Logistic Regression)']
[07:36:06] Full budget: curriculum variants = 8+32 epochs, single-phase variants = 40 epochs (early